# SermukaiBot · Avance de Proyecto 1
**Curso:** Desarrollo de aplicaciones con inteligencia artificial
**Estudiante:** Victor Javier Osuna Barajas · **Matrícula:** AL02689794
**Profesor:** Daniel Jiménez Alcantar
**Actividad:** Avance de Proyecto 1 · **Fecha de entrega:** 25 de octubre de 2026

Asistente basado en reglas con Python y NLTK que orienta a los empleados en el uso de la **plataforma de inventario Sermukai** (Assets, Stocks, Catálogo, Archivados, Mantenimientos, Ubicaciones y Auditoría). Es de carácter informativo: no consume APIs ni ejecuta acciones en el sistema.

## Cómo usar este notebook
1. Ejecuta las celdas en orden con el botón ▶ o **Entorno de ejecución → Ejecutar todas**.
2. Revisa las diez conversaciones automáticas y descarga sus evidencias en la celda de exportación.
3. Activa la conversación manual cuando quieras conversar con el asistente.
4. Guarda una copia en Drive o en GitHub (**Archivo → Guardar una copia en GitHub**).

Las pruebas automáticas son conversaciones simuladas; complementa sus registros con capturas de tu ejecución real.

## 1. Definición de la solución
**Problemática:** la empresa, que brinda guardias de seguridad a negocios públicos y privados, migró su inventario de archivos de Excel a un sistema web nuevo. Los empleados aún no dominan el sistema y la documentación formal tomará tiempo en estar lista.

**Objetivo:** ofrecer un chatbot con el conocimiento esencial de operación del sistema para acortar la curva de aprendizaje, reducir las interrupciones a supervisores y facilitar la incorporación de nuevos empleados.

**Usuarios:** empleados de la plataforma de inventario con alguno de los tres roles del sistema:

| Rol | Puede hacer |
|---|---|
| **Lector** | Consultar Dashboard, Assets, Stocks y Catálogo; buscar, filtrar y exportar. |
| **Operador** | Lo anterior, más crear/editar activos, stock, productos y ubicaciones; asignar, devolver, enviar a mantenimiento; archivar y restaurar. |
| **Administrador** | Todo lo anterior, más dar de baja activos, eliminar ubicaciones y consultar Auditoría. |

**Elementos inventariados:** radios (activos seriados → *Assets*) y uniformes, gorras y accesorios (consumibles por cantidad y talla → *Stocks*).

**Alcance:** orientación paso a paso sobre las pantallas del sistema. No realiza trámites, no consulta datos reales y no modifica el inventario.

## 2. Arquitectura conceptual
Usuario (botón «ChatBot» en la esquina inferior derecha) → entrada de texto → normalización → identificación de tema → clasificación de intención (motor de reglas NLTK ↔ base de conocimiento) → extracción del rol → respuesta informativa → confirmación → cierre.

A diferencia de un bot sin memoria, SermukaiBot conserva un **contexto mínimo por sesión**:
- `tema`: última sección mencionada (Assets, Catálogo, Ubicaciones…), para entender preguntas de seguimiento como «¿y cómo lo devuelvo?».
- `rol`: rol declarado por el usuario, para advertir cuando una acción no está disponible para su perfil.
- `pendiente`: última explicación dada, para ampliar la información si el usuario responde que no le sirvió.

## 3. Flujo conversacional
1. **Saludo inicial:** bienvenida y temas principales.
2. **Identificación de necesidad:** se detecta la sección (p. ej. «ubicaciones»).
3. **Clasificación de la intención:** dentro de la sección se identifica qué se quiere hacer (crear, editar, eliminar…). Si solo se menciona la sección, el bot muestra sus opciones.
4. **Extracción de datos relevantes:** si la acción requiere permisos, el bot pregunta el rol del usuario.
5. **Respuesta:** pasos informativos para que el usuario ejecute la acción.
6. **Confirmación:** el bot pregunta si la información sirvió; ante un «no» ofrece detalles y causas comunes.
7. **Cierre:** despedida.

In [ ]:
import sys
assert sys.version_info >= (3, 10), 'Se requiere Python 3.10 o superior.'
%pip -q install nltk==3.9.1
print('Python:', sys.version.split()[0])

## 4. Base de conocimiento y diseño conversacional
Cada intención contiene: identificador, patrón, rol mínimo requerido, respuesta y un detalle adicional que se muestra si la respuesta no fue suficiente. El orden importa: se utiliza la **primera coincidencia**, por eso las intenciones específicas (p. ej. *asignar un activo*) van antes que las generales (p. ej. *tema Assets*) y la regla `desconocida` va al final.

Los patrones se construyen con dos ayudas: `todas(...)` exige que aparezcan varios grupos de palabras en cualquier orden (acción + objeto) y `alguna(...)` basta con una palabra clave. Las respuestas usan los nombres reales de botones y campos de la aplicación.

| Intención | Frases de ejemplo | Rol mínimo |
|---|---|---|
| `asset_crear` | «¿Cómo creo un nuevo Asset?», «Tengo objetos en catálogo, ¿cómo los traslado a un Asset?» | Operador |
| `asset_asignar` | «¿Cómo le asigno una radio a un guardia?», «Ya tengo el Asset creado, ¿cómo lo entrego?» | Operador |
| `asset_mantenimiento` | «Un radio dejó de funcionar, ¿cómo lo marco?», «¿Cómo mando un equipo a mantenimiento?» | Operador |
| `asset_devolver` / `asset_baja` | «¿Cómo devuelvo un radio?», «¿Cómo doy de baja un equipo?» | Operador / Administrador |
| `catalogo_*` | «¿Cómo creo un producto?», «¿Diferencia entre Asset y Stock?», «¿Cómo archivo un producto?» | Lector–Operador |
| `stock_*` | «¿Cómo registro la entrega de uniformes?», «¿Qué significa stock bajo?» | Lector–Operador |
| `ubicacion_*` | «¿Cómo creo una ubicación?», «¿Cómo desactivo una bodega?» | Operador / Administrador |
| `mant_*` | «¿Cómo completo un mantenimiento?», «¿Cómo reprogramo un mantenimiento?» | Operador |
| `exportar`, `buscar`, `dashboard`, `auditoria` | «¿Cómo exporto a Excel?», «¿Cómo busco por número de serie?» | Lector / Administrador |

In [ ]:
import re
import unicodedata
import nltk
from nltk.chat.util import Chat, reflections


def todas(*grupos):
    """Exige que aparezcan todos los grupos de palabras, en cualquier orden."""
    return ''.join(rf'(?=.*\b{g}\b)' for g in grupos) + '.*'


def alguna(*grupos):
    """Basta con que aparezca una de las palabras o frases."""
    return r'.*\b(' + '|'.join(grupos) + r')\b.*'


def o(*patrones):
    """Combina patrones alternativos para una misma intención."""
    return '(?:' + '|'.join(f'(?:{p})' for p in patrones) + ')'


# Vocabulario del dominio (texto ya normalizado: minúsculas y sin acentos)
ASSET = r'(asset|assets|activo|activos|radio|radios|equipo|equipos)'
PERSONA = r'(guardia|guardias|empleado|empleados|elemento|elementos|persona)'
CATALOGO = r'(catalogo|catalogos|producto|productos|tipo de item|tipos de item)'
STOCK = r'(stock|stocks|uniforme|uniformes|gorra|gorras|accesorio|accesorios|consumible|consumibles|existencias|cantidad|talla|tallas)'
UBICACION = r'(ubicacion|ubicaciones|bodega|bodegas|almacen|almacenes|locacion|locaciones|sucursal|sucursales)'
MANTENIMIENTO = r'(mantenimiento|mantenimientos|reparacion|reparaciones|revision|taller)'
ROLES_TXT = r'(lector|lectora|reader|operador|operadora|worker|trabajador|trabajadora|almacenista|admin|administrador|administradora)'
CREAR = r'(crear|creo|crea|nuevo|nueva|registr\w*|alta|agreg\w*|anad\w*|traslad\w*|convert\w*|captur\w*)'
EDITAR = r'(edit\w*|modific\w*|cambi\w*|correg\w*|corrijo|actualiz\w*)'
ELIMINAR = r'(elimin\w*|borr\w*|quit\w*)'

BIENVENIDA = ('Bienvenido al bot de la plataforma de inventario, estoy para resolver tus dudas sobre el uso de la aplicación. '
              '¿Buen día, en qué te puedo ayudar el día de hoy? '
              'Temas principales: Assets, Stocks, Catálogo, Archivados, Mantenimientos, Ubicaciones, Exportar a Excel, Dashboard, Auditoría y Roles.')

# (intención, patrón, rol mínimo, respuesta, detalle si la respuesta no fue suficiente)
BASE = [
    ('vacio', r'^$', None, 'Escribe una consulta para poder orientarte.', None),
    ('despedida', r'^(salir|adios|hasta luego|bye|nos vemos)$', None,
     '¡Hasta luego! Gracias por consultar SermukaiBot. Vuelve cuando tengas otra duda sobre el sistema.', None),
    ('saludo', r'^(hola|hey|buenas|buen dia|buenos dias|buenas tardes|buenas noches|que tal)( (hola|buen dia|buenos dias|buenas tardes|buenas noches|que tal|bot|sermukaibot))*$', None,
     BIENVENIDA, None),
    ('rol_declarar', o(r'.*\b(soy|mi rol es|tengo (el )?rol( de)?|rol de)( un| una| el)? ' + ROLES_TXT + r'\b.*', ROLES_TXT + r'$'), None,
     'Gracias, tomaré en cuenta tu rol.', None),

    # --- Administración y ajustes generales ---
    ('auditoria', alguna('auditoria', 'historial', 'bitacora', r'quien (modifico|cambio|edito|borro)', 'registro de cambios'), 'administrador',
     'Abre Administración → Auditoría. Ahí se registra cada cambio del sistema con Fecha, Acción, Entidad, Usuario, Valores anteriores y Valores nuevos. '
     'Filtra por rango de fechas (accesos rápidos o «Desde»/«Hasta»), por «Entidad» y por «Acción»; usa «Limpiar filtros» para volver a ver todo.',
     'Si no ves Auditoría en la barra lateral es porque solo está disponible para Administradores. Si la tabla aparece vacía, revisa el rango de fechas: los filtros activos pueden estar ocultando registros.'),
    ('usuarios', alguna('usuarios', r'(crear|agregar|nuevo|alta de) usuario'), None,
     'La sección Administración → Usuarios aún está en construcción. Los accesos y roles (Lector, Operador, Administrador) los asigna el administrador del sistema en la cuenta corporativa de Microsoft (Entra ID).', None),
    ('sesion', alguna(r'(iniciar|inicio|cerrar|cierro) sesion', 'login', 'logout', 'contrasena', r'no puedo (entrar|ingresar)'), 'lector',
     'Para entrar pulsa «Iniciar sesión» en la pantalla de bienvenida y usa tu cuenta corporativa de Microsoft. Para salir, abre el menú de usuario en la parte inferior de la barra lateral y elige cerrar sesión.',
     'Si no puedes entrar, confirma que usas tu cuenta de la empresa y no una personal. Si entras pero no ves secciones, tu cuenta no tiene un rol asignado: pide al administrador que te asigne Lector, Operador o Administrador.'),
    ('apariencia', alguna('modo oscuro', 'modo claro', r'tema (oscuro|claro)', r'cambiar (el )?tema', 'dark mode'), None,
     'Puedes alternar entre tema claro y oscuro con el botón de tema que está en la parte inferior de la barra lateral. La preferencia se recuerda en tu navegador.', None),

    # --- Funciones transversales ---
    ('catalogo_importar', alguna(r'import\w*', 'carga masiva', r'duplic\w*', 'cambiar categoria'), None,
     'Los botones «Importar CSV», «Duplicar» y «Cambiar categoría» aparecen en Catálogo, pero todavía no están disponibles en esta versión. Por ahora registra los productos uno a uno con «Nuevo producto».', None),
    ('exportar', alguna(r'export\w*', r'descarg\w*', 'excel', 'xlsx', 'csv'), 'lector',
     'En Assets, Stocks o Catálogo pulsa «Exportar» y elige «Excel (.xlsx)» o «CSV (.csv)». '
     'En Assets y Catálogo también puedes marcar filas con las casillas y usar «Exportar selección» para descargar solo esas filas.',
     'La exportación incluye los registros de la sección. Si el archivo no se descarga, revisa que tu navegador permita descargas del sitio. CSV es texto plano compatible con cualquier hoja de cálculo; .xlsx abre directo en Excel.'),
    ('archivados_restaurar', alguna(r'restaur\w*', r'recuper\w*', r'desarchiv\w*'), 'operador',
     'Abre Inventario → Archivados. Ahí verás dos listas: productos del catálogo y stock fuera de uso. Usa «Restaurar» para regresar el registro a su sección o «Eliminar» si quieres borrarlo definitivamente.',
     'Archivados solo aparece para Operadores y Administradores. Usa el buscador de la página para encontrar el registro. Si restauras un producto, vuelve a aparecer en Catálogo y en las listas de selección.'),

    # --- Mantenimientos ---
    ('mant_completar', o(todas(r'(complet\w*|termin\w*|finaliz\w*)', MANTENIMIENTO), todas(r'(reparad[oa]s?|arreglad[oa]s?)')), 'operador',
     'Abre Operación → Mantenimientos, localiza el registro «Pendiente» y pulsa «Completar». Indica la «Fecha de finalización» (no puede ser futura) y notas opcionales, y confirma con «Completar». '
     'Si el activo ya no tiene otros mantenimientos pendientes, regresa a «En stock».',
     'Si el activo sigue «En reparación» después de completar, revisa si tiene otro mantenimiento pendiente en la bandeja. El botón «Completar» solo aparece en registros con estado Pendiente.'),
    ('mant_cancelar', o(todas(r'(cancel\w*)', MANTENIMIENTO), alguna(r'reprogram\w*', r'pospon\w*')), 'operador',
     'En Operación → Mantenimientos abre el registro pendiente. Para cambiar la fecha usa «Reprogramar», ajusta la «Fecha programada» y pulsa «Guardar». '
     'Para anularlo usa «Cancelar», escribe un motivo opcional y confirma con «Cancelar mantenimiento»; el activo vuelve a stock si no le quedan otros mantenimientos pendientes.',
     'La fecha programada no puede estar en el pasado. Solo se pueden reprogramar o cancelar mantenimientos con estado Pendiente.'),
    ('mant_ver', alguna('mantenimientos pendientes', r'pendientes de (mantenimiento|reparacion)', 'bandeja', r'(ver|veo|consulto) (los )?mantenimientos'), 'operador',
     'Abre Operación → Mantenimientos: es la bandeja de mantenimientos pendientes con las columnas Activo, Descripción, Programado, Creado por y Estado. '
     'El número junto al menú indica cuántos hay pendientes y el Dashboard también los muestra en la tarjeta «Mantenimientos pendientes».',
     'Haz clic en una fila para ver el detalle y sus notas. Usa el buscador para filtrar por número de serie o descripción.'),

    # --- Assets (activos seriados, p. ej. radios) ---
    ('asset_baja', alguna('de baja', 'baja definitiva', r'desech\w*', r'decomision\w*'), 'administrador',
     'Dar de baja retira un activo de forma definitiva. Abre Inventario → Assets, haz clic en el activo para ver su detalle, pulsa «Dar de baja» y confirma. '
     'Funciona con activos «En stock» o «En reparación»; si está «Asignado», primero debe devolverse.',
     'Si el botón aparece deshabilitado, el activo está asignado: usa «Devolver» antes. La baja no se puede deshacer, así que confirma el número de serie antes de aceptar.'),
    ('stock_bajo', alguna('stock bajo', 'bajo stock', 'poco stock', 'umbral', r'alerta (minima|de stock)', r'se (esta )?acab\w*', r'agotad\w*'), 'lector',
     'Un ítem muestra «Stock bajo» cuando su cantidad está por debajo de la alerta mínima («mín») definida para el producto. '
     'Lo verás en la columna Estado de Stocks y en la tarjeta «Stock bajo umbral» del Dashboard.',
     'Si un ítem no muestra «mín», no tiene alerta configurada y siempre aparecerá como OK. Para reponerlo, edita el ítem en Stocks y actualiza la «Cantidad».'),
    ('stock_cantidad', todas(r'(asign\w*|entreg\w*|repart\w*|dar|doy|salida|surt\w*|edit\w*|modific\w*|ajust\w*|actualiz\w*|cambi\w*|descont\w*|rest\w*|sum\w*)', STOCK), 'operador',
     'Los Stocks (uniformes, gorras, accesorios) se controlan por cantidad, no se asignan a personas como los Assets. '
     'Para registrar una entrega o una entrada, abre Inventario → Stocks, usa el menú «⋯» del ítem → «Editar», actualiza la «Cantidad» y pulsa «Guardar».',
     'Cada combinación de producto, talla y ubicación es un registro distinto: verifica que editas la talla y bodega correctas. La cantidad no puede ser negativa.'),
    ('asset_mantenimiento', o(todas(r'(mand\w*|envi\w*|llev\w*|report\w*|registr\w*|marc\w*|program\w*|nuevo|crear|agend\w*|meter|meto)', MANTENIMIENTO),
                              alguna(r'falla\w*', r'averi\w*', r'descompu\w*', 'dejo de funcionar', 'no funciona', 'no enciende', 'no prende', 'se rompio', r'rot[oa]', r'danad[oa]')), 'operador',
     'Para enviar un activo a mantenimiento abre Inventario → Assets, haz clic en el activo y pulsa «Enviar a mantenimiento» (o «Enviar a reparación» si está asignado). '
     'Escribe la «Descripción» de la falla, elige la «Fecha programada» y agrega notas si lo necesitas; confirma con «Enviar». '
     'El activo cambia a «En reparación» y aparece en Operación → Mantenimientos. También puedes hacerlo desde Mantenimientos → «Nuevo mantenimiento».',
     'Desde «Nuevo mantenimiento» solo se listan activos En stock; si el equipo está asignado hazlo desde su detalle en Assets. La fecha programada no puede estar en el pasado y la descripción es obligatoria.'),
    ('asset_asignar', todas(r'(asign\w*|entreg\w*|prest\w*|resguardo)', o(ASSET, PERSONA)), 'operador',
     'Para asignar un activo debe estar en estado «En stock». Abre Inventario → Assets, localízalo (buscador o filtro Estado) y haz clic en la fila para abrir su detalle. '
     'Pulsa «Asignar», escribe el «Nombre del empleado», opcionalmente elige la «Ubicación de servicio», ajusta la «Fecha de asignación» y, si aplica, una devolución esperada (+1, +3 o +6 meses). '
     'Confirma con «Asignar»: el estado cambiará a «Asignado».',
     'Si no aparece el botón «Asignar», el activo no está En stock (puede estar Asignado, En reparación o Dado de baja). Si ya está asignado a otra persona, primero usa «Devolver». El nombre del empleado es obligatorio.'),
    ('asset_devolver', todas(r'(devolver|devuelv\w*|devolucion|regres\w*|retorn\w*|recog\w*)', o(ASSET, PERSONA)), 'operador',
     'Abre Inventario → Assets y entra al detalle del activo con estado «Asignado». Pulsa «Devolver», indica la «Fecha de devolución» y notas opcionales, y confirma con «Devolver». El activo regresa a «En stock».',
     'El botón «Devolver» solo aparece en activos Asignados. En el detalle verás la sección «Asignación actual» con el nombre y la fecha de retorno esperada; si está en rojo, la devolución está vencida.'),
    ('asset_crear', todas(CREAR, ASSET), 'operador',
     'Para crear un Asset primero debe existir en Catálogo un producto con tipo de seguimiento «Serialized» (por número de serie). '
     'Después abre Inventario → Assets en la barra lateral y pulsa «Nuevo activo». En el modal elige el «Tipo de ítem» (solo aparecen productos seriados), '
     'captura el «Número de serie» y, si quieres, una foto y notas. Pulsa «Añadir» y el activo aparecerá en la tabla con estado «En stock».',
     'Si el producto no aparece en «Tipo de ítem», revisa en Catálogo que su tipo de seguimiento sea Serialized y que no esté archivado. Si no ves «Nuevo activo», tu rol es Lector. El número de serie es obligatorio.'),
    ('asset_editar', todas(EDITAR, ASSET), 'operador',
     'En Inventario → Assets abre el menú «⋯» de la fila (o el detalle del activo) y elige «Editar». Puedes cambiar el número de serie, la foto y las notas; pulsa «Guardar».',
     'El tipo de ítem no se puede cambiar al editar: si te equivocaste de producto, registra el activo correcto y da de baja el erróneo con ayuda de un Administrador.'),
    ('asset_estados', o(todas(r'(estado|estados|estatus|significa)', ASSET), alguna('en stock', 'asignado', 'en reparacion')), 'lector',
     'Un Asset puede estar: «En stock» (disponible para asignar), «Asignado» (entregado a un empleado o ubicación de servicio), «En reparación» (con mantenimiento pendiente) o «Dado de baja» (retirado definitivamente). '
     'Filtra por estado con el chip «Estado» en la tabla de Assets; el Dashboard muestra el conteo en «Activos por estado».',
     'Las acciones disponibles dependen del estado: En stock → Asignar, Enviar a mantenimiento, Dar de baja; Asignado → Devolver, Enviar a reparación; En reparación → se gestiona desde Mantenimientos.'),

    # --- Catálogo ---
    ('catalogo_tipos', alguna('serializado', 'seriado', 'serialized', 'quantity', 'por cantidad', 'tipo de seguimiento', 'seguimiento', 'diferencia'), 'lector',
     'El Catálogo define los tipos de producto y es la base de Assets y Stocks. Cada producto tiene un tipo de seguimiento: '
     '«Serialized» (se rastrea por número de serie y se gestiona en Assets, p. ej. radios) o «Quantity» (se maneja por cantidad en Stocks, p. ej. uniformes, gorras y accesorios).',
     'El tipo de seguimiento no se puede cambiar después de crear el producto, así que elígelo con cuidado. En el detalle del producto verás «Seriado» o «Cantidad».'),
    ('catalogo_crear', todas(CREAR, CATALOGO), 'operador',
     'Abre Inventario → Catálogo y pulsa «Nuevo producto». Llena Nombre, Categoría, Marca y Modelo (obligatorios), elige el «Tipo de seguimiento» (Serialized o Quantity) '
     'e indica si «¿Requiere mantenimiento?»; si es Sí, define el intervalo en días. Pulsa «Añadir».',
     'Elige Serialized para equipos con número de serie (radios) y Quantity para consumibles (uniformes, gorras). Este tipo no se podrá cambiar después.'),
    ('catalogo_archivar', todas(r'(archiv\w*)', CATALOGO), 'operador',
     'En Inventario → Catálogo abre el menú «⋯» del producto y elige «Archivar». El producto deja de aparecer en el catálogo y en las listas de selección, y se mueve a Inventario → Archivados, desde donde puedes restaurarlo.',
     'Archivar es la opción recomendada cuando un producto deja de usarse pero quieres conservar su historial.'),
    ('catalogo_eliminar', todas(ELIMINAR, CATALOGO), 'operador',
     'En Inventario → Catálogo usa el menú «⋯» del producto → «Eliminar», o marca varias filas y usa «Eliminar» en la barra de acciones en lote. '
     'Si solo quieres dejar de usarlo, es preferible «Archivar».',
     'Eliminar es definitivo. Si el producto ya tiene activos o stock relacionados, archívalo en lugar de eliminarlo.'),

    # --- Stocks (consumibles por cantidad) ---
    ('stock_crear', todas(CREAR, STOCK), 'operador',
     'Abre Inventario → Stocks y pulsa «Nuevo ítem». Elige el «Tipo de ítem» (productos de catálogo de tipo Quantity), la «Cantidad», la «Ubicación» y, si aplica, la «Talla». Pulsa «Añadir».',
     'Si el producto no aparece, revisa en Catálogo que sea de tipo Quantity. Si la bodega no aparece, revisa en Ubicaciones que esté activa.'),
    ('stock_archivar', todas(r'(archiv\w*|elimin\w*|borr\w*)', STOCK), 'operador',
     'En Inventario → Stocks abre el menú «⋯» del ítem y elige «Archivar» para sacarlo de uso (podrás restaurarlo desde Archivados) o «Eliminar» para borrarlo.',
     'Archivar conserva el registro; eliminar no se puede deshacer.'),

    # --- Ubicaciones ---
    ('ubicacion_eliminar', todas(ELIMINAR, UBICACION), 'administrador',
     'En Operación → Ubicaciones abre el menú «⋯» de la ubicación y elige «Eliminar». Esta acción solo está disponible para Administradores.',
     'Si la ubicación todavía tiene stock o activos, considera «Desactivar» en lugar de eliminarla.'),
    ('ubicacion_desactivar', todas(r'(desactiv\w*|inactiv\w*|activar|reactiv\w*)', UBICACION), 'operador',
     'En Operación → Ubicaciones abre el menú «⋯» de la ubicación y elige «Desactivar» (o «Activar» para reactivarla). La columna Estado mostrará «Activa» o «Inactiva».',
     'Las ubicaciones inactivas dejan de aparecer al registrar stock. Desactivar es reversible; eliminar no.'),
    ('ubicacion_crear', todas(CREAR, UBICACION), 'operador',
     'Abre Operación → Ubicaciones y pulsa «Nueva ubicación». Escribe el «Nombre» (obligatorio) y, opcionalmente, «Dirección» y «Descripción». Pulsa «Añadir». '
     'Las ubicaciones activas aparecen al registrar stock y como «Ubicación de servicio» al asignar activos.',
     'Si no ves Ubicaciones en la barra lateral, tu rol es Lector. Usa nombres claros (p. ej. «Bodega Centro») para encontrarlas fácilmente.'),

    # --- Consulta ---
    ('dashboard', alguna('dashboard', 'tablero', 'panel', 'resumen', 'indicadores', 'metricas', 'graficas'), 'lector',
     'El Dashboard (Principal → Dashboard) muestra tres tarjetas: «Activos por estado», «Stock bajo umbral» y «Mantenimientos pendientes».',
     'Si una tarjeta muestra «Error al cargar», recarga la página. Las cifras se actualizan al volver a la sección.'),
    ('buscar', alguna(r'busc\w*', r'filtr\w*', 'encontrar', 'encuentro', r'localiz\w*', 'numero de serie'), 'lector',
     'Cada sección tiene un buscador arriba de la tabla (en Assets busca por serie, tipo o notas) y chips de filtro como «Estado» o «Tipo de ítem». '
     'Para encontrar una sección, pulsa la tecla «/» y escribe su nombre en el buscador de la barra lateral.',
     'Si aparece «Sin resultados», usa «Limpiar filtros» o «Limpiar búsqueda». Puedes ordenar columnas haciendo clic en su encabezado.'),
    ('roles_info', alguna('rol', 'roles', 'permiso', 'permisos', 'acceso', 'privilegios'), None,
     'El sistema tiene tres roles: Lector (consulta y exporta), Operador (crea, edita, asigna, envía a mantenimiento, archiva) y Administrador (además da de baja activos, elimina ubicaciones y consulta Auditoría). '
     'Los asigna el administrador del sistema. Dime tu rol (por ejemplo «soy operador») para ajustar mis respuestas.', None),

    # --- Confirmación y cierre ---
    ('confirm_no', r'^(no|nop|nel|todavia no|aun no|sigo sin|tampoco)( .*)?$', None,
     'Cuéntame con más detalle qué intentas hacer y en qué sección del sistema estás.', None),
    ('confirm_si', r'^(si|sip|claro|listo|ya|funciono|me sirvio|perfecto|excelente|genial|ok|okay|vale|de acuerdo)( .*)?$', None,
     '¡Excelente! ¿Necesitas ayuda con otro tema del sistema?', None),
    ('agradecimiento', alguna('gracias', 'te agradezco', 'muy amable'), None,
     '¡De nada! ¿Necesitas orientación sobre otro tema?', None),
    ('ayuda', alguna('ayuda', 'menu', 'opciones', 'que puedes hacer', 'temas', 'que sabes'), None,
     'Puedo orientarte sobre: Assets (crear, asignar, devolver, mantenimiento, baja), Stocks (registrar, entregas, stock bajo), Catálogo (productos y tipos), '
     'Archivados, Mantenimientos, Ubicaciones, exportar a Excel, búsqueda, Dashboard, Auditoría y roles. Escribe salir para terminar.', None),

    # --- Tema mencionado sin acción: se muestran sus opciones ---
    ('tema_assets', alguna(ASSET), None,
     'Assets son los activos fijos rastreados por número de serie (p. ej. radios). ¿Qué necesitas: crear, asignar, devolver, enviar a mantenimiento, dar de baja, editar o conocer sus estados?', None),
    ('tema_catalogo', alguna(CATALOGO), None,
     'El Catálogo contiene los tipos de producto, base de Assets y Stocks. ¿Quieres crear, archivar o eliminar un producto, o conocer la diferencia entre Serialized y Quantity?', None),
    ('tema_stocks', alguna(STOCK), None,
     'Stocks son los ítems por cantidad (uniformes, gorras, accesorios). ¿Quieres registrar un ítem, actualizar cantidades por entregas, entender el stock bajo o archivar un ítem?', None),
    ('tema_ubicaciones', alguna(UBICACION), None,
     'Ubicaciones son los lugares físicos donde se guardan activos y stock. ¿Quieres crear, desactivar o eliminar una ubicación?', None),
    ('tema_mantenimientos', alguna(MANTENIMIENTO), None,
     'En Mantenimientos puedes enviar activos a reparación, ver la bandeja de pendientes, completar, reprogramar o cancelar. ¿Qué necesitas?', None),
    ('tema_archivados', alguna(r'archiv\w*'), None,
     'Archivados reúne productos de catálogo y stock fuera de uso. ¿Quieres archivar algo o restaurar un registro?', None),
    ('desconocida', r'.*', None,
     'No identifiqué tu consulta. Menciona una sección (Assets, Stocks, Catálogo, Mantenimientos, Ubicaciones) y lo que quieres hacer, o escribe «menú».', None),
]

NIVEL = {'lector': 1, 'operador': 2, 'administrador': 3}
SINONIMOS_ROL = {'lector': 'lector', 'lectora': 'lector', 'reader': 'lector',
                 'operador': 'operador', 'operadora': 'operador', 'worker': 'operador', 'trabajador': 'operador',
                 'trabajadora': 'operador', 'almacenista': 'operador',
                 'admin': 'administrador', 'administrador': 'administrador', 'administradora': 'administrador'}
RESUMEN_ROL = {
    'lector': 'Como Lector puedes consultar Dashboard, Assets, Stocks y Catálogo, buscar, filtrar y exportar, pero no crear ni modificar registros.',
    'operador': 'Como Operador puedes crear y editar activos, stock, productos y ubicaciones; asignar, devolver y enviar activos a mantenimiento; y archivar o restaurar registros.',
    'administrador': 'Como Administrador tienes todos los permisos del Operador y además puedes dar de baja activos, eliminar ubicaciones y consultar la Auditoría.',
}
# Palabra que se agrega a una pregunta de seguimiento para resolverla con el tema en contexto
PALABRA_TEMA = {'assets': 'activo', 'catalogo': 'producto', 'stocks': 'stock', 'ubicaciones': 'ubicacion',
                'mantenimientos': 'mantenimiento', 'archivados': 'archivados'}
PREFIJO_TEMA = {'asset_': 'assets', 'tema_assets': 'assets', 'catalogo_': 'catalogo', 'tema_catalogo': 'catalogo',
                'stock_': 'stocks', 'tema_stocks': 'stocks', 'ubicacion_': 'ubicaciones', 'tema_ubicaciones': 'ubicaciones',
                'mant_': 'mantenimientos', 'tema_mantenimientos': 'mantenimientos',
                'archivados_': 'archivados', 'tema_archivados': 'archivados'}
CONFIRMACION = ' ¿Te sirvió la información? Responde «sí» o «no».'

ROL_MINIMO = {nombre: rol for nombre, _, rol, _, _ in BASE}
DETALLE = {nombre: detalle for nombre, _, _, _, detalle in BASE}
chatbot = Chat([(patron, [respuesta]) for _, patron, _, respuesta, _ in BASE], reflections)


def normalizar(texto):
    texto = unicodedata.normalize('NFD', texto.lower().strip())
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


def nueva_sesion():
    return {'tema': None, 'rol': None, 'pendiente': None, 'rol_preguntado': False}


def detectar(limpio):
    return next(nombre for nombre, patron, *_ in BASE if re.match(patron, limpio, re.IGNORECASE))


def tema_de(intencion):
    return next((tema for prefijo, tema in PREFIJO_TEMA.items() if intencion.startswith(prefijo)), None)


def nota_rol(intencion, sesion):
    """Extracción de datos relevantes: ajusta la respuesta según el rol del usuario."""
    minimo = ROL_MINIMO[intencion]
    if minimo in (None, 'lector'):
        return ''
    rol = sesion['rol']
    if rol is None:
        if not sesion['rol_preguntado']:
            sesion['rol_preguntado'] = True
            return f' Esta acción requiere rol {minimo.capitalize()} o superior. ¿Cuál es tu rol: lector, operador o administrador?'
        return f' (Requiere rol {minimo.capitalize()} o superior.)'
    if NIVEL[rol] < NIVEL[minimo]:
        return (f' Atención: con tu rol de {rol.capitalize()} no verás esta opción; pide apoyo a un usuario '
                f'con rol {minimo.capitalize()} o solicita el permiso a tu administrador.')
    return ''


def responder(texto, sesion):
    limpio = normalizar(texto)
    intencion, contexto = detectar(limpio), False

    # Pregunta de seguimiento sin tema explícito: se completa con el tema en contexto
    if intencion == 'desconocida' and sesion['tema']:
        ampliado = f"{limpio} {PALABRA_TEMA[sesion['tema']]}"
        alterna = detectar(ampliado)
        if alterna != 'desconocida' and not alterna.startswith('tema_'):
            intencion, limpio, contexto = alterna, ampliado, True

    respuesta = chatbot.respond(limpio)

    if intencion == 'rol_declarar':
        sesion['rol'] = SINONIMOS_ROL[re.search(ROLES_TXT, limpio).group(1)]
        respuesta += ' ' + RESUMEN_ROL[sesion['rol']]
        if sesion['pendiente']:
            aviso = nota_rol(sesion['pendiente'], sesion)
            respuesta += aviso or ' Con tu rol sí puedes realizar la acción que consultaste.'
            respuesta += CONFIRMACION
    elif intencion == 'confirm_no' and sesion['pendiente']:
        respuesta = ('Revisemos con más detalle: ' + DETALLE[sesion['pendiente']] +
                     ' Si aún tienes dudas, contacta a tu supervisor o al administrador del sistema.')
        sesion['pendiente'] = None
    elif DETALLE[intencion]:
        aviso = nota_rol(intencion, sesion)
        respuesta += aviso
        if '¿Cuál es tu rol' not in aviso:
            respuesta += CONFIRMACION
        sesion['pendiente'] = intencion
    else:
        sesion['pendiente'] = None

    sesion['tema'] = tema_de(intencion) or sesion['tema']
    return {'intencion': intencion, 'respuesta': respuesta, 'contexto': contexto}


print(f'Intenciones registradas: {len(BASE)}')
for nombre, _, rol, _, _ in BASE:
    print(f'  {nombre:<22} rol mínimo: {rol or "—"}')
print('Versión NLTK:', nltk.__version__)

## 5. Diez conversaciones distintas de prueba
Incluyen las seis frases de entrada del diseño conversacional, saludo, declaración de rol, confirmación positiva y negativa, preguntas de seguimiento resueltas por contexto, acentos, mayúsculas, texto vacío, una consulta fuera de alcance y una consulta con dos temas. Esta última muestra la prioridad de reglas; no demuestra comprensión de ambas intenciones.

Las intenciones esperadas se declaran antes de ejecutar las pruebas. La tasa obtenida describe únicamente estos casos diseñados, no la precisión general ante usuarios reales. Cada conversación inicia una sesión nueva (sin tema ni rol).

In [ ]:
CONVERSACIONES = [
    # Flujo completo: saludo → necesidad → extracción del rol → confirmación → cierre
    [('Hola', 'saludo'), ('¿Cómo creo un nuevo Asset?', 'asset_crear'), ('Soy operador', 'rol_declarar'),
     ('Sí, ya quedó', 'confirm_si'), ('Gracias', 'agradecimiento')],
    # Confirmación negativa: el bot amplía la explicación
    [('¿Ya tengo diferentes objetos en catálogos, cómo puedo ahora trasladarlo a un Asset?', 'asset_crear'),
     ('No me sirvió', 'confirm_no'), ('Adiós', 'despedida')],
    # El rol limita la acción
    [('¿Cómo le asigno una radio a un guardia?', 'asset_asignar'), ('Soy lector', 'rol_declarar'), ('salir', 'despedida')],
    # Pregunta de seguimiento resuelta por contexto (tema Assets)
    [('Ya tengo el Asset creado, ¿cómo lo entrego a un guardia?', 'asset_asignar'), ('¿Y cómo lo devuelvo?', 'asset_devolver')],
    [('Un radio dejó de funcionar ¿Cómo lo marco en el sistema?', 'asset_mantenimiento'),
     ('Ya quedó reparado, ¿cómo lo regreso a stock?', 'mant_completar')],
    [('¿Cómo mando un equipo a mantenimiento?', 'asset_mantenimiento'), ('¿Cómo reprogramo un mantenimiento?', 'mant_cancelar')],
    # Tema sin acción → opciones; luego acciones resueltas por contexto
    [('Ubicaciones', 'tema_ubicaciones'), ('¿Cómo creo una nueva?', 'ubicacion_crear'), ('¿Y para eliminarla?', 'ubicacion_eliminar')],
    [('¿Cómo registro la entrega de uniformes?', 'stock_cantidad'), ('¿Qué significa stock bajo?', 'stock_bajo'),
     ('¿Cómo exporto la tabla a Excel?', 'exportar'), ('¿Puedo importar productos desde un CSV?', 'catalogo_importar')],
    # Texto vacío, mayúsculas, fuera de alcance y menú
    [('   ', 'vacio'), ('¿QUÉ DIFERENCIA HAY ENTRE ASSET Y STOCK?', 'catalogo_tipos'),
     ('¿Qué clima hará mañana?', 'desconocida'), ('menú', 'ayuda')],
    # Dos temas en una frase: gana la primera regla en la base (asignar antes que crear producto)
    [('Quiero crear un producto y asignarlo a un guardia', 'asset_asignar'),
     ('¿Quién modificó este registro? Necesito la auditoría', 'auditoria'), ('hasta luego', 'despedida')],
]

registros = []
for numero, conversacion in enumerate(CONVERSACIONES, 1):
    sesion = nueva_sesion()
    print(f'\n=== Conversación {numero:02d} ===')
    for entrada, esperada in conversacion:
        salida = responder(entrada, sesion)
        correcto = salida['intencion'] == esperada
        registros.append({'conversacion': numero, 'entrada': entrada, 'esperada': esperada, **salida,
                          'rol': sesion['rol'], 'correcto': correcto})
        print('Usuario:', repr(entrada))
        print('SermukaiBot:', salida['respuesta'])
        print('Intención:', salida['intencion'], '| Por contexto:', salida['contexto'],
              '| Resultado:', 'OK' if correcto else 'REVISAR')

aciertos = sum(r['correcto'] for r in registros)
print(f'\nResultado: {aciertos}/{len(registros)} turnos coinciden con lo esperado.')
assert aciertos == len(registros), 'Hay casos que requieren revisión.'

### Verificación del comportamiento conversacional
Además de la intención, se comprueba que el bot pregunte el rol, advierta cuando el rol no alcanza, amplíe la explicación ante un «no» y resuelva seguimientos por contexto.

In [ ]:
def turno(conversacion, entrada):
    return next(r for r in registros if r['conversacion'] == conversacion and r['entrada'] == entrada)

verificaciones = {
    'Pregunta el rol cuando la acción lo requiere': '¿Cuál es tu rol' in turno(1, '¿Cómo creo un nuevo Asset?')['respuesta'],
    'Confirma que el Operador puede crear Assets': 'sí puedes' in turno(1, 'Soy operador')['respuesta'],
    'Amplía la explicación ante un «no»': turno(2, 'No me sirvió')['respuesta'].startswith('Revisemos con más detalle'),
    'Advierte que el Lector no puede asignar': 'Atención' in turno(3, 'Soy lector')['respuesta'],
    'Resuelve «¿y cómo lo devuelvo?» por contexto': turno(4, '¿Y cómo lo devuelvo?')['contexto'],
    'Resuelve «¿y para eliminarla?» por contexto': turno(7, '¿Y para eliminarla?')['contexto'],
}
for descripcion, cumple in verificaciones.items():
    print('OK     ' if cumple else 'REVISAR', descripcion)
assert all(verificaciones.values()), 'Hay verificaciones que requieren revisión.'

## 6. Conversación manual opcional
Cambia `ACTIVAR_CHAT` a `True` y ejecuta la celda. Escribe `salir` para terminar. Se deja desactivada para que **Ejecutar todas** no quede esperando una entrada.

In [ ]:
ACTIVAR_CHAT = False
if ACTIVAR_CHAT:
    sesion = nueva_sesion()
    print('SermukaiBot:', BIENVENIDA)
    while True:
        entrada = input('Tú: ')
        salida = responder(entrada, sesion)
        print('SermukaiBot:', salida['respuesta'])
        if salida['intencion'] == 'despedida':
            break
else:
    print('Activa ACTIVAR_CHAT para conversar manualmente.')

## 7. Exportación de evidencias
Genera un archivo CSV y uno de texto con las respuestas obtenidas en esta ejecución y los empaqueta en ZIP. En Colab puedes activar su descarga. Captura además las salidas visibles del notebook para el documento de entrega.

In [ ]:
import csv
import zipfile
from pathlib import Path
from datetime import datetime, timezone

carpeta = Path('evidencias_sermukaibot')
carpeta.mkdir(exist_ok=True)
with (carpeta / 'pruebas.csv').open('w', newline='', encoding='utf-8-sig') as f:
    escritor = csv.DictWriter(f, fieldnames=list(registros[0]))
    escritor.writeheader()
    escritor.writerows(registros)
lineas = ['Ejecución UTC: ' + datetime.now(timezone.utc).isoformat(), f'Python: {sys.version.split()[0]} | NLTK: {nltk.__version__}']
for r in registros:
    lineas.extend([f"Conversación {r['conversacion']} | Usuario: {r['entrada']!r}", 'SermukaiBot: ' + r['respuesta'],
                   f"Intención: {r['intencion']} | Esperada: {r['esperada']} | Contexto: {r['contexto']} | Rol: {r['rol']} | Correcto: {r['correcto']}", ''])
(carpeta / 'conversaciones.txt').write_text('\n'.join(lineas), encoding='utf-8')
with zipfile.ZipFile('Evidencias_SermukaiBot.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for archivo in carpeta.iterdir():
        z.write(archivo, arcname=archivo.name)
print('Evidencias generadas: Evidencias_SermukaiBot.zip')
DESCARGAR = False  # Cambia a True si deseas descargar el ZIP en Colab.
if DESCARGAR:
    from google.colab import files
    files.download('Evidencias_SermukaiBot.zip')

## 8. Resultados, limitaciones y siguientes etapas
Completa el análisis después de ejecutar:
- ¿Cuántos turnos coincidieron con la intención esperada?
- ¿Qué ocurrió con la consulta sobre el clima y el texto vacío?
- ¿Por qué se respondió sobre asignar un activo ante «crear un producto y asignarlo a un guardia»?
- ¿Cómo cambió la respuesta de asignar un radio cuando el usuario dijo que era Lector?
- Prueba «assets» y después «¿cómo lo edito?». ¿Conserva el tema?

**Limitaciones:** coincidencias por palabras clave; prioridad por orden de reglas; el contexto solo conserva el último tema, el rol y la última explicación; no maneja negaciones ni errores ortográficos; el rol lo declara el usuario y no se valida contra su sesión real; la base de conocimiento debe actualizarse cuando cambie la interfaz (p. ej. cuando «Importar CSV» o «Usuarios» estén disponibles). Las diez conversaciones son pruebas controladas, no una evaluación representativa.

**Mejoras:** integrar el chatbot como widget «ChatBot» en la esquina inferior derecha de la aplicación Angular, leer el rol directamente del token de sesión, ampliar la base con el recorrido completo del *happy path* de cada sección, incorporar clasificación de intenciones con aprendizaje automático y recopilar preguntas reales de los empleados para medir la precisión.

## Referencias técnicas para consulta
- NLTK: https://www.nltk.org/api/nltk.chat.util.html
- Python `re`: https://docs.python.org/3/library/re.html
- Google Colab: https://colab.research.google.com/